# Stage 1 Validation — Feature Pipeline (`F`) -> Feast -> Credit Risk Training (`T`)

Interactive validation notebook for testing the Feature Pipeline (`F`), Feast Offline Feature Store, and PoC Model 1 (`credit_risk_batch` — LightGBM -> ONNX) Training Pipeline (`T`) inside Docker (`docker compose up --build -d`) or offline:

- **Cells 1–2** inspect the shared `feast_data` and `model_artifacts` volumes populated by `feast`, `feature`, and `train` without mutating them, and run `TrainingPipeline` against the containerized `FeatureStore`.
- **Cells 3–5** execute an isolated end-to-end `FeaturePipeline -> FeastFeatureStore -> TrainingPipeline -> ONNX Runtime` workflow in a sandbox directory, verifying causal point-in-time joins, convergence, ONNX parity (`< 1e-5`), cohort fairness, independent ONNX scoring, and `@champion` / `@previous` rotation.

In [1]:
import os
import tempfile
from datetime import UTC, datetime, timedelta
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd

from banking_fti.adapters import FeastFeatureStore, InMemoryModelRegistry
from banking_fti.core.contracts import (
    HOME_OWNERSHIP_VALUES,
    RAW_CARD_EVENTS_COLUMNS,
    RawBankingBatch,
)
from banking_fti.features.pipeline import FeaturePipeline
from banking_fti.training import (
    CreditRiskGateConfig,
    TabularPreprocessor,
    TrainingPipeline,
)

repo_path = Path(os.environ.get("FEAST_REPO_PATH", "feature_repo")).resolve()
shared_default = Path(tempfile.gettempdir()) / "banking_fti_feast_shared_train"
shared_data_dir = Path(os.environ.get("FEAST_DATA_DIR", str(shared_default))).resolve()
shared_store = FeastFeatureStore(repo_path=repo_path, data_dir=shared_data_dir)

shared_artifact_default = Path(tempfile.gettempdir()) / "banking_fti_model_artifacts_shared"
shared_artifact_dir = Path(
    os.environ.get("MODEL_ARTIFACT_DIR", str(shared_artifact_default))
).resolve()

sandbox_feast_default = Path(tempfile.gettempdir()) / "banking_fti_train_nb_feast"
sandbox_feast_dir = Path(
    os.environ.get("TRAINING_NOTEBOOK_SANDBOX_FEAST_DIR", str(sandbox_feast_default))
).resolve()
sandbox_store = FeastFeatureStore(repo_path=repo_path, data_dir=sandbox_feast_dir)

sandbox_art_default = Path(tempfile.gettempdir()) / "banking_fti_train_nb_artifacts"
sandbox_artifact_dir = Path(
    os.environ.get("TRAINING_NOTEBOOK_SANDBOX_ARTIFACT_DIR", str(sandbox_art_default))
).resolve()

registered_entities = sorted(entity.name for entity in shared_store.list_entities())
registered_views = sorted(view.name for view in shared_store.list_feature_views())
shared_counts = {
    group: len(shared_store.read_feature_group(group))
    for group in ("customer_profiles", "card_window_aggs", "credit_applications")
}
persisted_artifacts = (
    sorted(
        str(p.relative_to(shared_artifact_dir))
        for p in shared_artifact_dir.rglob("*")
        if p.is_file()
    )
    if shared_artifact_dir.exists()
    else []
)
print("Entities:", registered_entities)
print("FeatureViews:", registered_views)
print("Shared container row counts:", shared_counts)
print("Shared container model artifacts:", persisted_artifacts)

Entities: ['credit_card', 'customer', 'loan_application']
FeatureViews: ['card_window_aggs', 'card_window_aggs_by_customer', 'credit_applications', 'credit_applications_by_customer', 'customer_profiles']
Shared container row counts: {'customer_profiles': 3986, 'card_window_aggs': 106446, 'credit_applications': 2113}
Shared container model artifacts: ['credit_risk_batch/20261127T032346Z-f5b29d2e/model.onnx', 'credit_risk_batch/20261127T032346Z-f5b29d2e/model.txt', 'credit_risk_batch/20261127T032346Z-f5b29d2e/preprocessor.joblib']


In [2]:
if shared_counts["credit_applications"] > 0:
    shared_registry = InMemoryModelRegistry()
    shared_pipeline = TrainingPipeline(
        feature_store=shared_store,
        model_registry=shared_registry,
        artifact_dir=sandbox_artifact_dir / "shared_container_run",
        executor="jupyter_notebook_shared",
    )
    shared_train_result = shared_pipeline.train(num_boost_round=60, random_state=42)
    print("Shared container TrainingResult:", shared_train_result.to_dict())
else:
    shared_train_result = None
    print("Shared Feast store is empty (expected in isolated offline tests).")

Shared container TrainingResult: {'model_name': 'credit_risk_batch', 'gate_passed': False, 'holdout_auc': 0.6401091013494115, 'holdout_recall': 0.23255813953488372, 'brier': 0.1368026516294814, 'onnx_max_abs_diff': 9.116796095831603e-08, 'parity_pass': True, 'champion_version': None, 'previous_version': None, 'challenger_registered': True, 'challenger_version': 1, 'dataset_sha256': 'f5b29d2ee464e9985e6c629f0dea2f499ef4ac67a5c887059fb312ca93bbe0ff', 'promoted': False, 'rows_used': 2113, 'convergence_passed': True, 'nan_inf_passed': True, 'value_skew_passed': True, 'fairness_passed': False, 'serving_smoke_passed': True, 'champion_comparison_passed': True, 'gate_reasons': ['holdout roc_auc 0.640 < 0.75', 'holdout default_recall 0.233 < 0.65', 'cohort income_tier=high recall 0.184 < 0.5', 'cohort income_tier recall gap 0.416 > 0.25', 'cohort home_ownership=MORTGAGE recall 0.250 < 0.5', 'cohort home_ownership=OTHER recall 0.400 < 0.5', 'cohort home_ownership=OWN recall 0.214 < 0.5', 'cohort

/app/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
2026-11-27 03:25:21.677209153 [W:onnxruntime:, execution_frame.cc:943 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {529} for output label
2026-11-27 03:25:21.678838144 [W:onnxruntime:, execution_frame.cc:943 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {8} for output label


In [3]:
def build_separable_banking_batch(
    *, num_customers: int = 160, num_apps: int = 480, seed: int = 23
) -> RawBankingBatch:
    rng = np.random.default_rng(seed)
    base_ts = datetime(2026, 1, 1, tzinfo=UTC)
    tiers = np.array([35_000.0, 75_000.0, 130_000.0])
    incomes = np.array([tiers[i % 3] + rng.uniform(-5000.0, 5000.0) for i in range(num_customers)])
    homes = np.array([HOME_OWNERSHIP_VALUES[i % 4] for i in range(num_customers)])
    cust_ids = [f"NB-CUST-{i:04d}" for i in range(num_customers)]
    cc_nums = [f"40009999{i:08d}" for i in range(num_customers)]

    profiles_df = pd.DataFrame(
        {
            "customer_id": cust_ids,
            "event_timestamp": pd.Series(
                pd.to_datetime([base_ts] * num_customers, utc=True).as_unit("ns")
            ),
            "age_years": rng.integers(23, 68, size=num_customers, dtype=np.int64),
            "income_annual": incomes.astype(np.float64),
            "employment_years": rng.uniform(1.0, 25.0, size=num_customers).astype(np.float64),
            "home_ownership": homes,
            "tenure_months": rng.integers(6, 120, size=num_customers, dtype=np.int64),
            "num_active_cards": np.ones(num_customers, dtype=np.int64),
        }
    )

    card_rows = []
    for idx in range(num_customers):
        for step in range(3):
            card_rows.append(
                {
                    "event_id": f"NB-EVT-{idx:04d}-{step}",
                    "cc_num": cc_nums[idx],
                    "customer_id": cust_ids[idx],
                    "event_timestamp": base_ts + timedelta(hours=step + 1),
                    "amount": float(np.round(rng.uniform(25.0, 300.0), 2)),
                    "latitude": 40.7,
                    "longitude": -74.0,
                }
            )
    card_events_df = pd.DataFrame(card_rows).loc[:, list(RAW_CARD_EVENTS_COLUMNS)]
    card_events_df["event_timestamp"] = pd.Series(
        pd.to_datetime(card_events_df["event_timestamp"], utc=True).dt.as_unit("ns")
    )

    app_cust_indices = np.arange(num_apps) % num_customers
    app_ts = [base_ts + timedelta(hours=6 * (i + 1)) for i in range(num_apps)]
    dti = rng.uniform(0.05, 0.75, size=num_apps)
    util = rng.uniform(0.05, 0.90, size=num_apps)
    delinq = rng.integers(0, 4, size=num_apps, dtype=np.int64)
    pub_rec = (rng.random(num_apps) < 0.25).astype(np.int64)
    risk_score = 2.6 * dti + 1.8 * util + 0.65 * delinq + 0.9 * pub_rec - 2.35
    labels = (risk_score > 0.0).astype(np.int64)
    flip_mask = rng.random(num_apps) < 0.02
    labels = np.where(flip_mask, 1 - labels, labels).astype(np.int64)

    apps_df = pd.DataFrame(
        {
            "application_id": [f"NB-APP-{i:05d}" for i in range(num_apps)],
            "customer_id": [cust_ids[idx] for idx in app_cust_indices],
            "event_timestamp": pd.Series(pd.to_datetime(app_ts, utc=True).as_unit("ns")),
            "loan_amount": rng.uniform(5_000.0, 60_000.0, size=num_apps).astype(np.float64),
            "loan_term_months": rng.choice([12, 24, 36, 48, 60], size=num_apps).astype(np.int64),
            "dti_ratio": dti.astype(np.float64),
            "existing_delinquencies": delinq,
            "utilization_ratio": util.astype(np.float64),
            "pub_rec_flag": pub_rec,
            "default_label": labels,
        }
    )
    return RawBankingBatch(
        customer_profiles=profiles_df,
        card_events=card_events_df,
        credit_applications=apps_df,
        batch_id="nb-stage1-backfill",
    )


sandbox_batch = build_separable_banking_batch()
feature_res = FeaturePipeline(feature_store=sandbox_store).run_batch(sandbox_batch)
print(
    "Sandbox FeaturePipeline rows_written:",
    {group: receipt.rows_written for group, receipt in feature_res.receipts.items()},
)

sandbox_registry = InMemoryModelRegistry()
sandbox_pipeline = TrainingPipeline(
    feature_store=sandbox_store,
    model_registry=sandbox_registry,
    artifact_dir=sandbox_artifact_dir / "sandbox_run",
    executor="jupyter_notebook",
)
sandbox_result = sandbox_pipeline.train(num_boost_round=50, random_state=42)

assert sandbox_result.convergence_passed is True
assert sandbox_result.parity_pass is True
assert sandbox_result.onnx_max_abs_diff is not None and sandbox_result.onnx_max_abs_diff < 1e-5
assert sandbox_result.fairness_passed is True
assert sandbox_result.serving_smoke_passed is True
assert sandbox_result.gate_passed is True
assert sandbox_result.promoted is True

champion = sandbox_registry.get_champion("credit_risk_batch")
assert champion is not None and champion.version == 1
print(
    f"Promoted @champion v{champion.version}: AUC={sandbox_result.holdout_auc:.3f}, "
    f"Recall={sandbox_result.holdout_recall:.3f}, Brier={sandbox_result.brier:.3f}, "
    f"ONNX diff={sandbox_result.onnx_max_abs_diff:.2e}"
)

Sandbox FeaturePipeline rows_written: {'customer_profiles': 160, 'card_window_aggs': 480, 'credit_applications': 480}
Promoted @champion v1: AUC=0.954, Recall=0.857, Brier=0.088, ONNX diff=6.56e-08


/app/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
2026-11-27 03:25:29.109740577 [W:onnxruntime:, execution_frame.cc:943 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {120} for output label
2026-11-27 03:25:29.111307670 [W:onnxruntime:, execution_frame.cc:943 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {8} for output label


In [4]:
reloaded_pp = TabularPreprocessor.load(champion.preprocessor_path)
onnx_sess = ort.InferenceSession(Path(champion.onnx_model_path).read_bytes())

apps = sandbox_store.read_feature_group("credit_applications")
entity_df = apps.loc[:, ["application_id", "customer_id", "event_timestamp"]].tail(10)
inference_df = sandbox_store.get_historical_features(
    entity_df,
    ["customer_profiles", "card_window_aggs", "credit_applications"],
    exclude_labels=True,
)
X_infer = reloaded_pp.transform_features(inference_df)
onnx_outputs = onnx_sess.run(None, {"input": X_infer})
probs = np.asarray(onnx_outputs[1], dtype=np.float64)[:, 1]
onnx_scores_valid = bool(
    probs.shape == (len(entity_df),) and np.all((probs >= 0.0) & (probs <= 1.0))
)
assert onnx_scores_valid

preview = pd.DataFrame(
    {
        "application_id": inference_df["application_id"],
        "customer_id": inference_df["customer_id"],
        "default_probability": np.round(probs, 4),
        "predicted_default": (probs >= 0.5).astype(int),
        "model_version": champion.version,
        "feature_view_version": champion.feature_view_version,
    }
)
preview

2026-11-27 03:25:37.805324737 [W:onnxruntime:, execution_frame.cc:943 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label


,application_id,customer_id,default_probability,predicted_default,model_version,feature_view_version
0,NB-APP-00470,NB-CUST-0150,0.3936,0,1,v1
1,NB-APP-00471,NB-CUST-0151,0.0590,0,1,v1
2,NB-APP-00472,NB-CUST-0152,0.9544,1,1,v1
3,NB-APP-00473,NB-CUST-0153,0.8135,1,1,v1
4,NB-APP-00474,NB-CUST-0154,0.6583,1,1,v1
5,NB-APP-00475,NB-CUST-0155,0.9592,1,1,v1
6,NB-APP-00476,NB-CUST-0156,0.3512,0,1,v1
7,NB-APP-00477,NB-CUST-0157,0.9009,1,1,v1
8,NB-APP-00478,NB-CUST-0158,0.1565,0,1,v1
9,NB-APP-00479,NB-CUST-0159,0.9636,1,1,v1


In [5]:
second_pipeline = TrainingPipeline(
    feature_store=sandbox_store,
    model_registry=sandbox_registry,
    artifact_dir=sandbox_artifact_dir / "sandbox_run_2",
    gate_config=CreditRiskGateConfig(tie_tolerance=0.005),
    executor="jupyter_notebook",
)
second_result = second_pipeline.train(num_boost_round=50, random_state=42)
assert second_result.gate_passed is True
assert second_result.promoted is True
assert second_result.champion_version == 2
assert second_result.previous_version == 1

active_champion = sandbox_registry.get_champion("credit_risk_batch")
prior_previous = sandbox_registry.get_previous("credit_risk_batch")
assert active_champion is not None and prior_previous is not None
rotation_verified = active_champion.version == 2 and prior_previous.version == 1

print(
    "Active @champion version:",
    active_champion.version,
    "| @previous version:",
    prior_previous.version,
)
print("Lineage feature_schema_hash:", active_champion.feature_schema_hash[:16] + "...")
print("Lineage dataset_sha256:", active_champion.dataset_sha256[:16] + "...")
print("Rotation verified:", rotation_verified)

Active @champion version: 2 | @previous version: 1
Lineage feature_schema_hash: 5ea0cf6899f1d69c...
Lineage dataset_sha256: 5e6c5d5c4ac3219d...
Rotation verified: True


/app/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
2026-11-27 03:25:51.256671457 [W:onnxruntime:, execution_frame.cc:943 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {120} for output label
2026-11-27 03:25:51.258053622 [W:onnxruntime:, execution_frame.cc:943 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {8} for output label
